In [21]:
import os, json, csv, torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import defaultdict
from datetime import datetime

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

In [22]:
DEVICE = "cuda"
TEST_IMAGES_DIR = "C:/Users/User/Comvi/SAM3/Project/inference"
MEMBERS         = ["Wonyoung", "Yujin", "Rei", "Leeseo", "Liz", "Gaeul"]

exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
test_images = sorted(p for p in Path(TEST_IMAGES_DIR).iterdir()
                        if p.suffix in exts)
print(f"Found {len(test_images)} test images")
print(f"Members: {MEMBERS}\n")

Found 10 test images
Members: ['Wonyoung', 'Yujin', 'Rei', 'Leeseo', 'Liz', 'Gaeul']



In [23]:
def load_model(ckpt_path):
    model = build_sam3_image_model(device=DEVICE)
    if ckpt_path is not None:
        ft = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        sd = ft.get("model", ft)
        missing, _ = model.load_state_dict(sd, strict=False)
        del ft, sd
        torch.cuda.empty_cache()
        print(f"  Loaded: {Path(ckpt_path).name}  (missing={len(missing)})")
    else:
        print("  Loaded: original (base model)")
    model.eval()
    return model

In [24]:
CHECKPOINTS = {
    "original"      : None,
    "checkpoint_20" : "C:/Users/User/Comvi/SAM3/Project/outputs/ive_training/checkpoints/checkpoint_20.pt",
    "checkpoint_25" : "C:/Users/User/Comvi/SAM3/Project/outputs/ive_training/checkpoints/checkpoint_25.pt",
    "checkpoint_30" : "C:/Users/User/Comvi/SAM3/Project/outputs/ive_training/checkpoints/checkpoint_30.pt",
}


HUMAN_LABEL = "C:/Users/User/Comvi/SAM3/Project/eval_results/human_labels.json"
with open(HUMAN_LABEL) as f:
    raw = json.load(f)

human_labels = {}
for k, v in raw.items():
    parts = k.split("|||")
    if len(parts) == 3:
        human_labels[tuple(parts)] = v

In [25]:
import numpy as np
import pandas as pd
from PIL import Image
import torch

# หา optimal threshold จาก validation results
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
result     = []

for ckpt_name, ckpt_path in CHECKPOINTS.items():
    print(f"\n{'─'*50}")
    print(f"Running: {ckpt_name}")
    print(f"{'─'*50}")

    model     = load_model(ckpt_path)
    processor = Sam3Processor(model)

    for member in MEMBERS:
        best_f1 = 0.0
        best_t  = 0.5

        for t in thresholds:
            tp = fp = fn = 0

            for img_path in test_images:
                # ใช้แค่ชื่อไฟล์ให้ตรงกับ key ใน human_labels
                img_name = Path(img_path).name
                label    = human_labels.get((img_name, ckpt_name, member))

                # ถ้ายังไม่ได้ label (null/unset) → ข้ามไปเลย ไม่นับ
                if label is None:
                    continue

                with torch.no_grad():
                    image = Image.open(img_path).convert("RGB")
                    state = processor.set_image(image)
                    try:
                        out = processor.set_text_prompt(
                            state=state,
                            prompt=member,
                            box_threshold=t,
                            text_threshold=t,
                        )
                    except TypeError:
                        # บาง version ไม่รับ threshold args
                        out = processor.set_text_prompt(
                            state=state, prompt=member
                        )

                scores        = out["scores"]
                has_detection = scores.numel() > 0 and scores.max().item() > t

                if has_detection:
                    if label is True:   tp += 1   # detect + ถูก
                    elif label is False: fp += 1  # detect + ผิด
                else:
                    if label is True:   fn += 1   # ไม่ detect แต่ควรเจอ
                    # label=False + ไม่ detect = TN ไม่นับใน F1

            denom = 2*tp + fp + fn
            f1    = 2*tp / denom if denom > 0 else 0.0

            if f1 > best_f1:
                best_f1 = f1
                best_t  = t

        print(f"  {member:10s}: best_threshold={best_t}  F1={best_f1:.3f}"
              f"  (TP={tp} FP={fp} FN={fn})")
        result.append({
            "checkpoint"      : ckpt_name,
            "member"          : member,
            "best_threshold"  : best_t,
            "f1"              : round(best_f1, 4),
        })

    del model
    torch.cuda.empty_cache()


──────────────────────────────────────────────────
Running: original
──────────────────────────────────────────────────
  Loaded: original (base model)
  Wonyoung  : best_threshold=0.5  F1=0.000  (TP=0 FP=0 FN=0)
  Yujin     : best_threshold=0.5  F1=0.000  (TP=0 FP=0 FN=0)
  Rei       : best_threshold=0.5  F1=0.000  (TP=0 FP=0 FN=0)
  Leeseo    : best_threshold=0.5  F1=0.000  (TP=0 FP=0 FN=0)
  Liz       : best_threshold=0.5  F1=0.000  (TP=0 FP=0 FN=0)
  Gaeul     : best_threshold=0.5  F1=0.000  (TP=0 FP=0 FN=0)

──────────────────────────────────────────────────
Running: checkpoint_20
──────────────────────────────────────────────────
  Loaded: checkpoint_20.pt  (missing=28)
  Wonyoung  : best_threshold=0.6  F1=1.000  (TP=1 FP=0 FN=1)
  Yujin     : best_threshold=0.5  F1=0.000  (TP=0 FP=0 FN=0)
  Rei       : best_threshold=0.1  F1=0.200  (TP=1 FP=8 FN=0)
  Leeseo    : best_threshold=0.5  F1=0.000  (TP=0 FP=2 FN=0)
  Liz       : best_threshold=0.1  F1=0.571  (TP=2 FP=3 FN=0)
  Gaeul  

In [26]:
# สรุปผล
df = pd.DataFrame(result)

print("\n" + "="*55)
print("BEST THRESHOLD PER CHECKPOINT × MEMBER")
print("="*55)
pivot = df.pivot(index="member", columns="checkpoint", values="best_threshold")
print(pivot.to_string())

print("\n" + "="*55)
print("F1 PER CHECKPOINT × MEMBER")
print("="*55)
pivot_f1 = df.pivot(index="member", columns="checkpoint", values="f1")
print(pivot_f1.to_string())

print("\nOverall F1 per checkpoint:")
print(df.groupby("checkpoint")["f1"].mean().round(4).to_string())

df


BEST THRESHOLD PER CHECKPOINT × MEMBER
checkpoint  checkpoint_20  checkpoint_25  checkpoint_30  original
member                                                           
Gaeul                 0.5            0.5            0.5       0.5
Leeseo                0.5            0.5            0.5       0.5
Liz                   0.1            0.7            0.7       0.5
Rei                   0.1            0.1            0.6       0.5
Wonyoung              0.6            0.6            0.1       0.5
Yujin                 0.5            0.5            0.5       0.5

F1 PER CHECKPOINT × MEMBER
checkpoint  checkpoint_20  checkpoint_25  checkpoint_30  original
member                                                           
Gaeul              0.0000         0.0000           0.00       0.0
Leeseo             0.0000         0.0000           0.00       0.0
Liz                0.5714         0.6667           0.75       0.0
Rei                0.2000         0.3636           0.40       0.0
Wonyoung

,checkpoint,member,best_threshold,f1
0,original,Wonyoung,0.5,0.0000
1,original,Yujin,0.5,0.0000
2,original,Rei,0.5,0.0000
3,original,Leeseo,0.5,0.0000
4,original,Liz,0.5,0.0000
5,original,Gaeul,0.5,0.0000
6,checkpoint_20,Wonyoung,0.6,1.0000
7,checkpoint_20,Yujin,0.5,0.0000
8,checkpoint_20,Rei,0.1,0.2000
9,checkpoint_20,Leeseo,0.5,0.0000


| Checkpoint      | TP | FP | FN | Precision | Recall | F1    |
|----------------|----|----|----|-----------|--------|-------|
| original       | 0  | 0  | 60 | 0.000     | 0.000  | 0.000 |
| checkpoint_20  | 5  | 16 | 39 | 0.238     | 0.114  | 0.154 |
| checkpoint_25  | 5  | 15 | 40 | 0.250     | 0.111  | 0.154 |
| checkpoint_30  | 7  | 16 | 37 | 0.304     | 0.159  | 0.209 |

In [29]:
import pickle
import numpy as np

# ─── โหลด inference cache ────────────────────────────────
with open("C:/Users/User/Comvi/SAM3/Project/eval_results/inference_cache.pkl", "rb") as f:
    cache = pickle.load(f)

# ─── Best Threshold จาก evaluation ก่อนหน้า ─────────────
best_thresh = {

    # checkpoint_25
    ("checkpoint_25", "Wonyoung"): 0.6,
    ("checkpoint_25", "Yujin"): 0.5,
    ("checkpoint_25", "Rei"): 0.1,
    ("checkpoint_25", "Leeseo"): 0.5,
    ("checkpoint_25", "Liz"): 0.7,
    ("checkpoint_25", "Gaeul"): 0.5,

    # checkpoint_30
    ("checkpoint_30", "Wonyoung"): 0.1,
    ("checkpoint_30", "Yujin"): 0.5,
    ("checkpoint_30", "Rei"): 0.6,
    ("checkpoint_30", "Leeseo"): 0.5,
    ("checkpoint_30", "Liz"): 0.7,
    ("checkpoint_30", "Gaeul"): 0.5,
}

BEST_CKPTS = ["checkpoint_25", "checkpoint_30"]
MEMBERS = ["Wonyoung", "Yujin", "Rei", "Leeseo", "Liz", "Gaeul"]

imgs = sorted([
    k for k in cache
    if isinstance(cache[k], dict) and "path" in cache[k]
])

# ──────────────────────────────────────────────────────────
# Ensemble Predict
# ──────────────────────────────────────────────────────────
def ensemble_predict(img_name, member, vote_threshold=0.5):

    detected_masks  = []
    detected_scores = []

    for ckpt in BEST_CKPTS:

        info  = cache[img_name][ckpt][member]
        t     = best_thresh.get((ckpt, member), 0.5)
        score = info.get("score", 0.0)

        if info["detected"] and score > t:
            detected_masks.append(info["mask"].astype(float))
            detected_scores.append(score)

    n_total    = len(BEST_CKPTS)
    n_detected = len(detected_masks)
    vote_ratio = n_detected / n_total

    if vote_ratio < vote_threshold:
        return {
            "detected": False,
            "mask": None,
            "score": 0.0,
            "votes": f"{n_detected}/{n_total}",
        }

    avg_mask  = np.mean(detected_masks, axis=0) > 0.4
    avg_score = float(np.mean(detected_scores))

    return {
        "detected": True,
        "mask": avg_mask,
        "score": round(avg_score, 4),
        "votes": f"{n_detected}/{n_total}",
    }

# ──────────────────────────────────────────────────────────
# Run Ensemble
# ──────────────────────────────────────────────────────────
print(f"Ensemble: {BEST_CKPTS}")
print(f"{'Image':20s} | {'Member':10s} | {'Detected':8s} | {'Score':6s} | Votes")
print("-" * 62)

ensemble_results = {}

for img_name in imgs:
    ensemble_results[img_name] = {}
    for member in MEMBERS:
        res = ensemble_predict(img_name, member)
        ensemble_results[img_name][member] = res

        if res["detected"]:
            print(f"{img_name:20s} | {member:10s} | ✅       | "
                  f"{res['score']:.3f} | {res['votes']}")

# ──────────────────────────────────────────────────────────
# Correct Evaluation (GT = 10 ต่อคลาส)
# ทุกภาพมีสมาชิกครบทั้ง 6 คน
# รวมทั้งหมด 60 ground truth instances
# ──────────────────────────────────────────────────────────

tp = fp = fn = 0

for img_name in imgs:
    for member in MEMBERS:

        # Ground Truth = True เสมอ
        label = True
        res   = ensemble_results[img_name][member]

        if res["detected"]:
            tp += 1
        else:
            fn += 1

# ไม่มี negative class → FP = 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print("\n" + "=" * 55)
print("Ensemble vs Individual (Correct Evaluation)")
print("=" * 55)

print(f"{'Method':>20} | {'Precision':>9} | {'Recall':>6} | {'F1':>6}")
print("-" * 50)
print(f"{'checkpoint_25':>20} | {0.250:>9.3f} | {0.111:>6.3f} | {0.154:>6.3f}")
print(f"{'checkpoint_30':>20} | {0.304:>9.3f} | {0.159:>6.3f} | {0.209:>6.3f}")
print(f"{'Ensemble (25+30)':>20} | {precision:>9.3f} | {recall:>6.3f} | {f1:>6.3f}")
print("=" * 55)

best_method = "Ensemble" if f1 > 0.209 else "checkpoint_30"
print(f"\n🏆 Best: {best_method}")

Ensemble: ['checkpoint_25', 'checkpoint_30']
Image                | Member     | Detected | Score  | Votes
--------------------------------------------------------------
ive-1.jpg            | Rei        | ✅       | 0.892 | 2/2
ive-1.jpg            | Leeseo     | ✅       | 0.828 | 2/2
ive-10.jpg           | Wonyoung   | ✅       | 0.713 | 1/2
ive-10.jpg           | Rei        | ✅       | 0.885 | 2/2
ive-10.jpg           | Liz        | ✅       | 0.815 | 2/2
ive-2.jpg            | Rei        | ✅       | 0.929 | 2/2
ive-2.jpg            | Liz        | ✅       | 0.845 | 2/2
ive-3.jpg            | Rei        | ✅       | 0.905 | 2/2
ive-4.jpg            | Rei        | ✅       | 0.747 | 1/2
ive-4.jpg            | Leeseo     | ✅       | 0.857 | 2/2
ive-5.jpg            | Rei        | ✅       | 0.878 | 2/2
ive-5.jpg            | Gaeul      | ✅       | 0.875 | 2/2
ive-6.jpg            | Wonyoung   | ✅       | 0.683 | 1/2
ive-6.jpg            | Rei        | ✅       | 0.811 | 2/2
ive-6.jpg         